In [ ]:
#####################################################
#
# APLICAR PCA A DATOS QUE NUNCA HA VISTO
#
#####################################################
# Deben cargarse los archivos
# - preprocessor_cat.joblib
# - pca_pipe_num.joblib
# - pca_metadata.json
# - T_train_final_objetivo.csv
# - csv de tus nuevos datos
# Devolverá T_new_final.csv: el csv de PCA aplicado a los nuevos datos
#####################################################

# === Cargar artefactos para inferencia ===
import joblib
import json
import pandas as pd

preprocessor_cat = joblib.load("preprocessor_cat.joblib")
pca_pipe = joblib.load("pca_pipe_num.joblib")
with open("pca_metadata.json", "r") as f:
    meta = json.load(f)

cols_num = meta["cols_num"]
cols_cat = meta["cols_cat"]
pc_cols  = meta["pc_cols"]
cat_out_cols = meta["cat_out_cols"]

# PCA DE ENTRENAMIENTO CON OBJETIVO
#####################################################
#####################################################
entrenamiento = pd.read_csv("T_train_final_objetivo.csv")   ### todas las pca, categoricas y objetivo
entrenamiento_pca_objetivo = entrenamiento[pc_cols]
#####################################################
#####################################################

# NUEVOS DATOS
#####################################################
#####################################################
new_df = pd.read_csv("diabetes_desconocido.csv")
#####################################################
#####################################################
X_new_num = new_df[cols_num]
X_new_cat = new_df[cols_cat]

# Categóricas (mismo encoder, sin re-ajustar)
X_new_cat_proc = preprocessor_cat.transform(X_new_cat)
df_new_cat_encode = pd.DataFrame(X_new_cat_proc, columns=cat_out_cols, index=new_df.index)

# Numéricas → (preproc_num + StdScaler + PCA) con el pipeline guardado
T_new = pca_pipe.transform(X_new_num)
T_new_df = pd.DataFrame(T_new, columns=pc_cols, index=new_df.index)

# Final
T_new_final = pd.concat([T_new_df, df_new_cat_encode], axis=1)
T_new_final.to_csv("T_new_final.csv",index=False)

In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# --- 1) figura base: train coloreado por objetivo (igual que antes) ---
y_train = entrenamiento["objetivo"]
df_plot2 = entrenamiento_pca_objetivo.iloc[:, :2].copy()
df_plot2["objetivo"] = y_train.loc[entrenamiento_pca_objetivo.index].astype(str)

fig = px.scatter(
    df_plot2, x="PC1", y="PC2", color="objetivo",
    opacity=0.85, title="PC1 vs PC2 — Train (objetivo) + Nuevos",
    height=550
)
fig.update_traces(marker=dict(size=6))

# --- 2) overlay: nuevos (un solo trazo, símbolo distinto) ---
new2 = T_new_df.loc[:, ["PC1", "PC2"]].copy()
fig.add_trace(
    go.Scatter(
        x=new2["PC1"], y=new2["PC2"],
        mode="markers",
        name="nuevos",
        marker=dict(size=9, line=dict(width=1)),
        opacity=0.95,
        showlegend=True
    )
)

fig.update_layout(xaxis_title="PC1", yaxis_title="PC2")
fig.show()


In [ ]:
import plotly.express as px
import plotly.graph_objects as go

# base: train 3D
df3 = entrenamiento_pca_objetivo.iloc[:, :3].copy()
df3["objetivo"] = y_train.loc[entrenamiento_pca_objetivo.index].astype(str)

fig3 = px.scatter_3d(
    df3, x="PC1", y="PC2", z="PC3",
    color="objetivo", opacity=0.85,
    title="PC1–PC2–PC3 — Train (objetivo) + Nuevos",
    height=600
)
fig3.update_traces(marker=dict(size=5))

# overlay: nuevos 3D
new3 = T_new_df.loc[:, ["PC1","PC2","PC3"]].copy()
fig3.add_trace(
    go.Scatter3d(
        x=new3["PC1"], y=new3["PC2"], z=new3["PC3"],
        mode="markers",
        name="nuevos",
        marker=dict(size=6, line=dict(width=1)),
        opacity=0.95,
        showlegend=True
    )
)

fig3.update_layout(scene=dict(xaxis_title="PC1", yaxis_title="PC2", zaxis_title="PC3"))
fig3.show()
